# Pinecone 예제 01: 3차원 벡터 검색

첨부 파일의 `파인콘 벡터DB` 예제를 독립 실행 노트북으로 분리했습니다.

실습 흐름:
1. `.env`에서 `PINECONE_API_KEY`를 읽습니다.
2. `embedding-3d` 인덱스를 생성하거나 기존 인덱스를 재사용합니다.
3. `vec1`부터 `vec6`까지 3차원 벡터를 업서트합니다.
4. 필터 검색, 전체 ID 조회, fetch, 3D 시각화까지 확인합니다.


In [ ]:
import sys
print(sys.executable)


## 1. 패키지 설치

`pinecone-client`는 예전 패키지이므로 최신 `pinecone` 패키지와 충돌하지 않도록 제거합니다.


In [ ]:
%pip uninstall -y pinecone-client
%pip install -qU pinecone python-dotenv matplotlib


## 2. 환경변수와 Pinecone 클라이언트


In [ ]:
from dotenv import load_dotenv
from pinecone import Pinecone, ServerlessSpec
from itertools import chain
import os
import time

load_dotenv()

PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
if not PINECONE_API_KEY:
    raise ValueError("PINECONE_API_KEY is missing from your .env file.")

pc = Pinecone(api_key=PINECONE_API_KEY)

PINECONE_CLOUD = os.getenv("PINECONE_CLOUD", "aws")
PINECONE_REGION = os.getenv("PINECONE_REGION", "us-east-1")
INDEX_NAME = os.getenv("PINECONE_TOY_INDEX", "embedding-3d")
NAMESPACE = "embedding-3d-ns1"
VECTOR_DIMENSION = 3


## 3. 인덱스 목록과 상태 확인

첨부 파일은 바로 `create_index()`를 호출하지만, 같은 이름의 인덱스가 이미 있으면 오류가 납니다. 그래서 목록을 먼저 확인하고, 없을 때만 생성하도록 보완했습니다.


In [ ]:
def list_index_names():
    indexes = pc.list_indexes()
    if hasattr(indexes, "names"):
        return indexes.names()
    return [idx.get("name") if isinstance(idx, dict) else idx.name for idx in indexes]


def get_index_dimension(index_name):
    description = pc.describe_index(index_name)
    if isinstance(description, dict):
        return description.get("dimension")
    return getattr(description, "dimension", None)


def wait_until_ready(index_name, timeout_seconds=120):
    start = time.time()
    while time.time() - start < timeout_seconds:
        description = pc.describe_index(index_name)
        status = description.get("status") if isinstance(description, dict) else getattr(description, "status", {})
        ready = status.get("ready") if isinstance(status, dict) else getattr(status, "ready", False)
        if ready:
            return True
        time.sleep(3)
    raise TimeoutError(f"Index was not ready in time: {index_name}")


def ensure_index(index_name, dimension, metric="cosine"):
    indexes = list_index_names()
    print("indexes:", indexes)

    if index_name not in indexes:
        pc.create_index(
            name=index_name,
            dimension=dimension,
            metric=metric,
            spec=ServerlessSpec(cloud=PINECONE_CLOUD, region=PINECONE_REGION),
        )
        wait_until_ready(index_name)
    else:
        existing_dimension = get_index_dimension(index_name)
        if existing_dimension != dimension:
            raise ValueError(
                f"Index '{index_name}' has dimension {existing_dimension}, "
                f"but this example expects {dimension}."
            )
    return pc.Index(index_name)

index = ensure_index(INDEX_NAME, VECTOR_DIMENSION)
index.describe_index_stats()


## 4. 샘플 벡터 업서트


In [ ]:
vectors = [
    {"id": "vec1", "values": [1.0, 1.5, 2.0], "metadata": {"genre": "drama"}},
    {"id": "vec2", "values": [2.0, 1.0, 0.5], "metadata": {"genre": "action"}},
    {"id": "vec3", "values": [0.1, 0.3, 0.5], "metadata": {"genre": "drama"}},
    {"id": "vec4", "values": [1.0, 2.5, 3.5], "metadata": {"genre": "action"}},
    {"id": "vec5", "values": [3.0, 1.2, 1.3], "metadata": {"genre": "action"}},
    {"id": "vec6", "values": [0.3, 1.1, 2.5], "metadata": {"genre": "drama"}},
]

index.upsert(vectors=vectors, namespace=NAMESPACE)
time.sleep(2)

print(index.describe_index_stats())
for ids in index.list(namespace=NAMESPACE):
    print(ids)


## 5. 필터 검색

첨부 파일과 동일하게 쿼리 벡터 `[1.1, 0.3, 0.7]`을 사용하고, `genre`가 `drama`인 벡터만 검색합니다.


In [ ]:
query_vector = [1.1, 0.3, 0.7]

response = index.query(
    namespace=NAMESPACE,
    vector=query_vector,
    top_k=3,
    include_values=True,
    include_metadata=True,
    filter={"genre": {"$eq": "drama"}},
)
print(response)

assert len(response.matches) > 0, "No search results."
assert all(match.metadata["genre"] == "drama" for match in response.matches), "Filter result contains a non-drama vector."


## 6. 전체 벡터 fetch

첨부 파일의 `list()`와 `fetch()` 예제를 그대로 살려서, 저장된 벡터 값을 다시 가져옵니다.


In [ ]:
all_ids = list(chain.from_iterable(index.list(namespace=NAMESPACE)))
resp = index.fetch(ids=all_ids, namespace=NAMESPACE)
vec_map = resp.vectors

for vector_id, vector_data in vec_map.items():
    print(vector_id, vector_data.values, vector_data.metadata)

ids = list(vec_map.keys())
values = [vector_data.values for vector_data in vec_map.values()]
ids.append("qv")
values.append(query_vector)

print(ids)
print(values)


## 7. 3차원 시각화

저장된 벡터와 쿼리 벡터가 3차원 공간에서 어디에 있는지 확인합니다.


In [ ]:
import matplotlib.pyplot as plt

fig = plt.figure()
ax = fig.add_subplot(111, projection="3d")

for i, vector_id in enumerate(ids):
    if vector_id == "qv":
        ax.scatter(values[i][0], values[i][1], values[i][2], label=vector_id, color="y", s=50, marker="s")
    else:
        ax.scatter(values[i][0], values[i][1], values[i][2], label=vector_id)
    ax.text(values[i][0] + 0.1, values[i][1] + 0.1, values[i][2] + 0.1, vector_id)

ax.set_xlabel("X")
ax.set_ylabel("Y")
ax.set_zlabel("Z")
ax.set_title("Embedding Vector Space")
plt.show()

print("example 01 validation passed")
